# 01 · What Is MCP?

**Module 11 · MCP Tool Servers**

In Module 06 we saw that every framework wanted our tools **in its own way**: the Agents SDK, LangGraph and CrewAI each needed a different wrapper around the same functions. And every new app (a chat app, an IDE, a second team's agent) would need the same work again.

**MCP (Model Context Protocol)** is a standard plug for AI tools, like **USB-C**. You write your tools **once**, as a *server*. Any AI app that speaks MCP can discover and use them.

```mermaid
flowchart LR
    subgraph Without["Without a standard: M apps x N tools"]
        A1["App 1"] --- T1["Tool A"]
        A1 --- T2["Tool B"]
        A2["App 2"] --- T1
        A2 --- T2
    end
    subgraph With["With MCP: M apps + N servers"]
        B1["App 1"] --> P(["MCP"])
        B2["App 2"] --> P
        P --> S1["Server A"]
        P --> S2["Server B"]
    end
```

| Notebook | What you will learn |
|---|---|
| 1 | **What is MCP?** (this one): the roles, and the protocol itself, by talking to a server by hand |
| 2 | **Build a server**: our shop tools as an MCP server, with tools, resources and prompts |
| 3 | **Use it from an agent**: our own agent loop and the OpenAI Agents SDK, using the **same** server |
| 4 | **MCP security**: a poisoned third-party server, and the defences |

> The server in this module never calls a model. It only answers requests, so this notebook costs nothing.

## 1. The three roles
| Role | What it is | In our project |
|---|---|---|
| **Host** | the AI app the user talks to | our support desk |
| **Client** | the connector inside the host, one per server | `toolbox.py` |
| **Server** | a program that offers capabilities | `shop_server.py` |

A server can offer three kinds of things:

| Kind | What it is | Who decides to use it |
|---|---|---|
| **Tools** | actions: `lookup_order`, `issue_refund` | the **model** |
| **Resources** | data to read: the policies, one order | the **app** |
| **Prompts** | ready-made prompt templates | the **user** (for example from a menu) |

And two ways to connect (**transports**): **stdio** (the app starts the server as a child process and talks over its standard input and output: local, simple) and **HTTP** (a server running somewhere else: remote). This module uses stdio.

## 2. MCP is just JSON over a pipe
There is no magic. Let us talk to our server **by hand**, with nothing but `subprocess` and `json`. Messages are **JSON-RPC**: one JSON object per line. First we start the server as a child process:

In [1]:
import json
import subprocess
import sys

server = subprocess.Popen(
    [sys.executable, "shop_server.py"],
    stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True)


def send(message):
    """Write one JSON message to the server's input."""
    server.stdin.write(json.dumps(message) + "\n")
    server.stdin.flush()


def receive():
    """Read one JSON message from the server's output."""
    return json.loads(server.stdout.readline())


print("Server started, process id", server.pid)

Server started, process id 18876


### Step 1: the handshake
The client says hello and what it can do. The server answers with its own name and capabilities:

In [2]:
send({"jsonrpc": "2.0", "id": 1, "method": "initialize",
      "params": {"protocolVersion": "2025-06-18", "capabilities": {}, "clientInfo": {"name": "my-notebook", "version": "0.1"}}})

reply = receive()
print(json.dumps(reply, indent=2))

send({"jsonrpc": "2.0", "method": "notifications/initialized"})   # "I am ready" (a notification: no answer expected)

{
  "jsonrpc": "2.0",
  "id": 1,
  "result": {
    "capabilities": {
      "prompts": {
        "listChanged": false
      },
      "resources": {
        "listChanged": false,
        "subscribe": false
      },
      "tools": {
        "listChanged": false
      }
    },
    "instructions": "Tools for the support desk of an online shop: look up orders, check and issue refunds, search the policies.",
    "protocolVersion": "2025-06-18",
    "serverInfo": {
      "name": "shop-support",
      "version": ""
    }
  }
}


### Step 2: discovery. "What tools do you have?"
This is the heart of MCP. The client does **not** know the tools in advance. It asks:

In [3]:
send({"jsonrpc": "2.0", "id": 2, "method": "tools/list", "params": {}})
reply = receive()

for tool in reply["result"]["tools"]:
    print(tool["name"])

print()
print("The full description of one tool, exactly as it travels:")
print(json.dumps(reply["result"]["tools"][0], indent=2))

lookup_order
check_refund_eligibility
issue_refund
search_policies
list_refunds

The full description of one tool, exactly as it travels:
{
  "annotations": {
    "readOnlyHint": true
  },
  "description": "Get the details of one order: item, price, status and delivery date. The order id looks like A1001.",
  "inputSchema": {
    "properties": {
      "order_id": {
        "title": "Order Id",
        "type": "string"
      }
    },
    "required": [
      "order_id"
    ],
    "type": "object",
    "title": "lookup_orderArguments"
  },
  "name": "lookup_order",
  "outputSchema": {
    "properties": {
      "result": {
        "title": "Result",
        "type": "string"
      }
    },
    "required": [
      "result"
    ],
    "type": "object",
    "title": "lookup_orderOutput"
  }
}


Each tool comes with a **name**, a **description** (written for the model) and an **input schema** (JSON Schema, so the model knows what arguments to send). The server built all of this from a plain Python function. We look at how in notebook 2.

### Step 3: call a tool

In [4]:
send({"jsonrpc": "2.0", "id": 3, "method": "tools/call",
      "params": {"name": "lookup_order", "arguments": {"order_id": "A1001"}}})
print(json.dumps(receive(), indent=2))

send({"jsonrpc": "2.0", "id": 4, "method": "tools/call",
      "params": {"name": "issue_refund", "arguments": {"order_id": "A1004"}}})
print(json.dumps(receive()["result"]["content"], indent=2))

{
  "jsonrpc": "2.0",
  "id": 3,
  "result": {
    "content": [
      {
        "text": "{\"item\": \"Wireless headphones\", \"price\": 79.0, \"status\": \"delivered\", \"delivered_on\": \"2026-11-04\", \"days_since_delivery\": 18}",
        "type": "text"
      }
    ],
    "isError": false,
    "structuredContent": {
      "result": "{\"item\": \"Wireless headphones\", \"price\": 79.0, \"status\": \"delivered\", \"delivered_on\": \"2026-11-04\", \"days_since_delivery\": 18}"
    }
  }
}
[
  {
    "text": "The refund of $250.00 is above the automatic limit and now waits for a person. It has NOT been issued.",
    "type": "text"
  }
]


In [5]:
server.terminate()     # stop the child process
print("Server stopped")

Server stopped


That is the whole protocol for tools: **initialize, list, call**. The second call is worth a second look: asking for a refund of the $250 television did not issue it. The **server** applies the $100 limit, in code. Whichever app connects, the rule travels with the tools.

## 3. The same thing with the official client
Nobody writes JSON by hand. The `mcp` package has a client that does the handshake, the listing and the calls:

In [6]:
from toolbox import open_client, run_sync


async def ask_the_server():
    async with open_client("shop_server.py") as client:
        listed = await client.list_tools()
        names = []
        for tool in listed.tools:
            names.append(tool.name)
        print("Tools:", names)

        result = await client.call_tool("lookup_order", {"order_id": "A1002"})
        print("Order A1002:", result.content[0].text)


run_sync(ask_the_server)

Tools: ['lookup_order', 'check_refund_eligibility', 'issue_refund', 'search_policies', 'list_refunds']


Order A1002: {"item": "Running shoes", "price": 120.0, "status": "in_transit", "expected_delivery": "2026-11-26"}


(`run_sync` runs the async function for us. A notebook cannot start child processes from its own event loop on Windows. In a normal script you simply write `asyncio.run(ask_the_server())`.)

## 4. MCP versus the function calling we already know
| | Function calling (Module 02) | MCP |
|---|---|---|
| Where is the tool described? | in **your app's code**, for each model | in the **server**, once |
| Who runs the tool? | your app | the server (a separate program) |
| Adding a tool | change your app | change the server: clients just discover it |
| Reuse across apps | copy and paste | any MCP app |

MCP does not replace function calling. Underneath, the model still sees a list of tools and picks one. MCP is the **standard way to supply that list and run the call**.

## 5. When to use it, and when not
- **Use it** when several apps or teams need the same tools, when you want to use tools other people publish, or when you want tools you can test and run **separately** from the agent.
- **Skip it** for a single app with two functions. Plain function calling is simpler, and every moving part is a thing that can break.
- **It is not security.** MCP says how to connect. It does not say whom to trust. Notebook 4 is about that.

## What you learned
- MCP is a standard plug: **host, client, server**, offering **tools, resources and prompts**.
- Under the hood it is **JSON-RPC over a pipe**: initialize, list, call.
- The client **discovers** tools at run time. It contains no tool descriptions.

**Next:** build a server.